In [1]:
!pip3 install -q sentence-transformers faiss-cpu langchain langchain-openai pillow pymupdf


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [47]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

if os.environ["OPENAI_API_KEY"] is None:
    raise ValueError("OPENAI_API_KEY environment variable not set.")
else:
    print("OPENAI_API_KEY environment variable is set.")


OPENAI_API_KEY environment variable is set.


In [ ]:
from sentence_transformers import SentenceTransformer
from PIL import Image
import numpy as np

clip_model = SentenceTransformer("clip-ViT-B-32") ##https://huggingface.co/sentence-transformers/clip-ViT-B-32

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 896.87it/s, Materializing param=visual_projection.weight]                                
CLIPModel LOAD REPORT from: /Users/cardinality/.cache/huggingface/hub/models--sentence-transformers--clip-ViT-B-32/snapshots/327ab6726d33c0e22f920c83f2ff9e4bd38ca37f/0_CLIPModel
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [11]:
print("Dimensionality of sentence embeddings:", clip_model.get_sentence_embedding_dimension())

Dimensionality of sentence embeddings: None


In [10]:
len(clip_model.encode(["Hello world!"])[0])

512

In [15]:
import fitz  # PyMuPDF
import io
from pathlib import Path


PDF_PATH = Path("/Users/cardinality/Documents/BIA GIAI/bia-gndy-jan-genai/RAG/2510.14592v1.pdf")

In [20]:
all_text = []
doc = fitz.open(PDF_PATH)
for page_number in range(len(doc)):
    page = doc[page_number]
    text = page.get_text().strip()
    # Only add non-empty text to the list
    if text:
        all_text.append((text,"text",{"Source":"PDF", "page":{page_number + 1}}))
    # Extract images from the page
    for img_index, img_info in enumerate(page.get_images()):
        xref = img_info[0]
        try:
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
            all_text.append((image,"image",{"Source":"PDF", "page":{page_number + 1}}))
        except Exception as e:
            pass
doc.close()
print(f"Extracted {len(all_text)} text and image elements from the PDF.")
    

Extracted 18 text and image elements from the PDF.


In [23]:
all_text

[('Multimodal RAG for Unstructured Data:\nLeveraging Modality-Aware Knowledge Graphs\nwith Hybrid Retrieval\nRashmi R1 and Vidyadhar Upadhya2\n1 National Institute of Technology Karnataka, Surathkal, India,\nrashmir.243cd003@nitk.edu.in,\n2 National Institute of Technology Karnataka, Surathkal,\nIndia\nAbstract. Current Retrieval-Augmented Generation (RAG) systems pri-\nmarily operate on unimodal textual data, limiting their effectiveness\non unstructured multimodal documents. Such documents often combine\ntext, images, tables, equations, and graphs, each contributing unique in-\nformation. In this work, we present a Modality-Aware Hybrid retrieval\nArchitecture (MAHA), designed specifically for multimodal question an-\nswering with reasoning through a modality-aware knowledge graph. MAHA\nintegrates dense vector retrieval with structured graph traversal, where\nthe knowledge graph encodes cross-modal semantics and relationships.\nThis design enables both semantically rich and context-

In [27]:
vectors_list = []

for content,kind,metadata in all_text:
    if kind == "text":
        vector = clip_model.encode(content, convert_to_numpy=True)
        vectors_list.append((vector))
    elif kind == "image":
        vector = clip_model.encode(content,convert_to_numpy= True)
        vectors_list.append((vector))

In [29]:
vectors = np.array(vectors_list, dtype=np.float32)
print(f"Created a numpy array of shape {vectors.shape} containing the vector representations of text and images.")

Created a numpy array of shape (18, 512) containing the vector representations of text and images.


In [30]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_core.documents import Document
import base64
import tempfile


In [31]:
d = vectors.shape[1]  # Dimensionality of the vectors
faiss_index = faiss.IndexFlatIP(d)  # IP distance index
faiss_index.add(vectors)

In [33]:
docstore_dict = {}

for i, (content,kind,meta) in enumerate(all_text):
    if kind == "text":
        page_content = content
    else:
        tmp = tempfile.NamedTemporaryFile(suffix=".png",delete=False)
        content.save(tmp.name)
        page_content = tmp.name
    docstore_dict[str(i)] = Document(page_content=page_content, metadata={"kind":kind, **meta} )


docstore = InMemoryDocstore()
docstore.add(docstore_dict)
print("FAISS index  and docstore ready . Number of entries :",len(docstore_dict))

        

FAISS index  and docstore ready . Number of entries : 18


In [82]:
question = "Are there any reported metrics, scores, or statistics from the charts? List them briefly."
k=5

In [83]:
query_vector = clip_model.encode(question,convert_to_numpy=True).astype(np.float32)
query_vector = query_vector.reshape(1,-1)
query_vector.shape

(1, 512)

In [84]:
distances, indices =faiss_index.search(query_vector,k)

In [85]:
retrived_docs =[]
for idx in indices[0]:
    doc_id = str(idx)
    doc = docstore.search(doc_id)
    if doc :
        retrived_docs.append(doc)

In [86]:
retrived_docs

[Document(metadata={'kind': 'text', 'Source': 'PDF', 'page': {2}}, page_content='2\nRashmi R et al.\nthe forefront of this paradigm shift is Retrieval-Augmented Generation (RAG),\na powerful technique that augments the generative capacity of LLMs by en-\nabling them to fetch and incorporate relevant information from external data\nsources dynamically. This dynamic mechanism is crucial for mitigating “halluci-\nnations”(the generation of factually incorrect or fabricated information), thereby\nproducing accurate and trustworthy responses.\nIn RAGs, the retriever plays a central role in identifying relevant information\nfrom large collections of documents. Different types of retrievers bring distinct\nstrengths. Sparse retrievers, such as Best Matching-25 (BM25) [10], rely on lexi-\ncal overlap to locate documents containing exact query terms. They are efficient\nand effective for keyword-driven searches, often serving as an initial filtering\nstep. However, because they lack semantic aw

In [87]:
from pathlib import Path

text_parts = []
content_for_llm =[]


for doc in retrived_docs:
    if doc.metadata.get("kind") == "image":
        path = doc.page_content
        if Path(path).exist:
            with open(path, "rb") as f:
                b64 = base64.standard_b64decode(f.read()).decode()
            content_for_llm.append({"type":"image_url","image_url":{ "url":f"data:image/png;base64,{b64}"}})
    else:
        text_parts.append(doc.page_content)


context_text = "\n\n".join(text_parts) if text_parts else "(no text context)"

prompt_text = f"use the following context  and images(optional) below to answer the question. \n \n Context :\n {context_text} \n\n Question :\n {question}\n]n Answer it briefly"


content_for_llm.insert(0,{"type":"text","text":prompt_text})

In [88]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

message  = HumanMessage(content = content_for_llm)
llm= ChatOpenAI(model = "gpt-4o")

In [89]:
response = llm.invoke([message])
response.content

"The reported metrics and scores mentioned in the context include:\n\n1. **Improvement in Ranking Quality:** The precision of ranking quality is highlighted with an improvement of around 21% and 19% due to the incorporation of full modality coverage.\n\n2. **Modality Coverage Metric:** This metric evaluates the system's capability to retrieve evidence across multiple modalities. It is defined as the average coverage of modalities required by the ground truth answer across all queries, where a score of 1.0 indicates complete modality retrieval.\n\n3. **Performance Metrics:**\n   - **ROUGE-L:** A metric used to evaluate the quality of the proposed system, but specific values aren't given in the context.\n   - **Recall@3:** Another evaluation metric mentioned, though specific scores are not provided.\n   - **MRR (Mean Reciprocal Rank):** Also noted as a metric for evaluating the system's performance without specific scores mentioned."